# NLP Early Warning Framework

This notebook builds a monthly early-warning watchlist from ODI complaint narratives. The goal is to surface emerging and recurring cohort-topic signals that may deserve reviewer attention.

The main workflow main prepares complaint text, fits a topic model, labels the resulting topics, and then turns the complaint-topic assignments into monthly watchlist tables.

## 1. Setup

Import the packages, set the repo paths, and define the shared constants used throughout the notebook.


In [ ]:
# Import the packages used to clean complaint text, build topics, and score monthly watchlists
import contextlib
import html
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

RANDOM_SEED = 42

# Resolve repo paths so the notebook works from repo root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'

# Current exploratory inputs from consolidated preprocessing
MULTI_PATH = PROCESSED_DATA_DIR / 'odi_component_multilabel_cases.parquet'
SIDECAR_PATH = PROCESSED_DATA_DIR / 'odi_component_text_sidecar.parquet'
NLP_PATH = PROCESSED_DATA_DIR / 'odi_nlp_prepped.parquet'

## 2. Load The Component And Text Inputs

Load the processed multi-label component cases and the complaint text sidecar created during preprocessing. These two inputs give us the complaint cohorts, severity/context flags, and the narrative text needed for NLP.


In [ ]:
# Load the kept multi-label complaint cases and the complaint text sidecar, then merge them at the complaint level
multi_df = pd.read_parquet(MULTI_PATH)
sidecar_df = pd.read_parquet(SIDECAR_PATH)

sidecar_df['nlp_raw'] = sidecar_df['cdescr']
sidecar_df['nlp_base'] = sidecar_df['cdescr_model_text'].fillna('')
sidecar_df['raw_word_count'] = sidecar_df['cdescr_word_count']

# Make complaints that start with "attached document" placeholders as well
sidecar_df.loc[sidecar_df['nlp_base'].str[:25].str.contains(r'\battached\s+(document|documents)\b', case=False, regex=True), 'cdescr_placeholder_flag'] = True

side_cols = [
    'odino',
    'raw_word_count',
    'nlp_raw',
    'nlp_base',
    'cdescr_missing_flag',
    'cdescr_placeholder_flag'
]

sidecar_df = sidecar_df[side_cols].copy()

multi_cols = [
    'odino',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'ldate',
    'state',
    'severity_primary_flag',
    'severity_broad_flag',
    'component_groups'
]

multi_df = multi_df[multi_cols].copy()

df = multi_df.merge(sidecar_df, on='odino', how='inner', validate='one_to_one')
df['ldate'] = pd.to_datetime(df['ldate'])
df['month'] = df['ldate'].dt.to_period('M').astype(str)

df = df.loc[
    ~df['cdescr_missing_flag'] &
    ~df['cdescr_placeholder_flag'] &
    df['nlp_base'].str.strip().ne('')
].copy()

df.shape

C:\Users\davis\AppData\Local\Temp\ipykernel_928\3790629048.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sidecar_df.loc[sidecar_df['nlp_base'].str[:25].str.contains(r'\battached\s+(document|documents)\b', case=False, regex=True), 'cdescr_placeholder_flag'] = True


(345079, 15)

## 3. Build The NLP-Ready Complaint Table

Create one complaint-level table for NLP work. This section cleans the narrative text, keeps a few operational context flags, and marks obvious administrative or placeholder text so it can be handled carefully later.


In [ ]:
# Turn the raw complaint narrative into a modeling-ready text field and keep flags that explain major cleaning decisions

# Freedom of Information Act text patterns
FOIA_START_RE = re.compile(
    r"""
    (
        parts?\s+of\s+this\s+document\s+have\s+been\s+redacted
        |
        redacted\s+to\s+protect\s+personally\s+identifiable\s+information
        |
        information\s+redacted\s+pursuant\s+to\s+the\s+freedom\s+of\s+information\s+act
        |
        pursuant\s+to\s+the\s+free
        |
        freedom\s+of\s+information\s+act\s*\(?foia\)?
        |
        \bfoia\b\s*,?\s*5\s*u\.?s\.?c
    )
    """,
    flags=re.I | re.X
)

# NHTSA reviewer markers/updates
LEADING_MARKER_RE = re.compile(r'(^|(?<=[\s.;:]))[A-Z]{1,4}\*\s*:?', flags=re.I)
TRAILING_MARKER_RE = re.compile(r'\*[A-Z]{1,4}\b', flags=re.I)
UPDATED_RE = re.compile(r'\bUPDATED\s+\d{1,2}/\d{1,2}/\d{2,4}\.?', flags=re.I)

# Redacted placeholders, emails, URLS, phone numbers, and VIN numbers
PERSONAL_ARTIFACT_PATTERNS = [
    (re.compile(r'\[?X{2,}\]?', flags=re.IGNORECASE), ' '),
    (re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b'), ' '),
    (re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE), ' '),
    (re.compile(r'\b\d{3}[-.)\s]*\d{3}[-.\s]*\d{4}\b'), ' '),
    (re.compile(r'\b[A-HJ-NPR-Z0-9]{17}\b', flags=re.IGNORECASE), ' '),
]

# Repeated third party narratives and complaint form questions that don't add defect signal
ADMIN_PHRASES_RE = re.compile(
    r"""
    \b(
        consumer\s+writes\s+in\s+regards\s+to |
        (consumer|contact|owner|driver|complainant)\s+(?:also\s+)?(states|stated|has|had|owns|owned|leased)(?:\s+that)? |
        (consumer|contact|owner|driver|complainant)\s+(received|called|indicated|referenced|related|took|sent) |
        (?:please\s+)?see\s+(?:the\s+)?attached(?:\s+(?:document|documents|complaint|file|files|statement|letter|pdf))?(?:\s+(?:for|with)\s+complaint)? |
        vehicle\s+(was|is|had) |
        manufacturer\s+was\s+notified |
        dealer\s+was\s+notified |
        (?:what\s+)?component\s+or\s+system\s+failed\s+or\s+malfunctioned\s |
        (?:is\s+it\s+)?available\s+for\s+inspection\s+upon\s+request |
        how\s+was\s+your\s+safety\s+or\s+the\s+safety\s+of\s+others\s+put\s+at\s+risk |
        has\s+the\s+problem\s+been\s+reproduced(?:\s+or\s+confirmed(?:\s+by\s+(?:an?\s+)?(?:independent\s+)?dealer(?:\s+or\s+(?:independent\s+)?(?:service\s+center|shop))?)?)? |
        (?:has\s+the\s+)?(?:vehicle\s+or\s+)?component\s+been\s+inspected(?:\s+by\s+the\s+manufacturer(?:,\s+police,\s+insurance\s+representatives(?:,)?\s+or\s+others)?)? |
        were\s+there\s+any\s+warning\s+(lamps|lights) |
        (?:warning\s+)?messages\s+or\s+(?:other\s+)?symptoms |
        problem\s+prior\s+to\s+the\s+failure(?:,\s+and\s+when\s+did\s+the\s+first\s+appear) |
        failure\s+mileage |
        approximate\s+failure\s+mileage |
        vehicle\s+was\s+not\s+diagnosed |
        vehicle\s+was\s+not\s+repaired |
        not\s+diagnosed\s+or\s+repaired |
        not\s+diagnosed\s+nor\s+repaired |
        manufacturer\s+was\s+not\s+notified |
        dealer\s+was\s+not\s+contacted |
        local\s+dealer\s+was\s+not\s+contacted |
        warning\s+light(?:s)?\s+(?:was\s+)?illuminated |
        while\s+driving\s+at\s+an\s+undisclosed\s+speed |
        while\s+driving\s+at\s+an\s+unknown\s+speed |
        at\s+an\s+undisclosed\s+speed |
        at\s+an\s+unknown\s+speed |
        the\s+vehicle\s+was\s+taken\s+to\s+the\s+local\s+dealer |
        the\s+vehicle\s+was\s+taken\s+to\s+an\s+independent\s+mechanic |
        the\s+vehicle\s+was\s+taken\s+to\s+the\s+dealer |
        where\s+it\s+was\s+diagnosed\s+that |
        where\s+it\s+was\s+determined\s+that |
        needed\s+to\s+be\s+replaced |
        was\s+diagnosed\s+that |
        was\s+determined\s+that |
        campaign\s+number |
  )\b[\s,:;-]*
    """,
    flags=re.IGNORECASE | re.VERBOSE
)

ADMIN_SENTENCE_ARTIFACTS_RE = re.compile(
    r"""
    (
        (?:^|(?<=[.!?]\s))
        (?:TL\*\s*)?
        the\s+contact\s+own(?:s|ed)\s+an?\s+[^.!?]{2,140}[.!?] |
        \b(?:the\s+)?(?:approximate\s+)?failure\s+mileage\s+(?:was\s+)?(?:approximately\s+)?(?:unknown|unavailable|not\s+available|[\d,]+)[.!?]? |
        \bthe\s+approximate\s+failure\s+mileage\s+was\s+unknown[.!?]? |
        \b(?:the\s+)?contact\s+received\s+notifications?\s+of\s+(?:the\s+)?nhtsa\s+campaign\s+numbers?[:\s]+[0-9A-Z,\sand]+[^.!?]{0,260}? |
        \b(?:the\s+)?contact\s+stated\s+that\s+the\s+manufacturer\s+exceeded\s+a\s+reasonable\s+amount\s+of\s+time\s+for\s+the\s+repair[.!?]? |
        \bvin\s+tool\s+confirms\s+parts?\s+(?:are\s+)?not\s+available[.!?]? |
        \bparts?\s+distribution\s+disconnect[.!?]? |
        \b(?:the\s+)?contact\s+had\s+not\s+experienced\s+(?:a\s+)?failure[.!?]? |
        \b(?:the\s+)?contact\s+had\s+not\s+experienced\s+the\s+failure[.!?]? |
        \b(?:the\s+)?(?:local\s+|unknown\s+|undisclosed\s+)?dealer\s+(?:was\s+)?(?:contacted|made\s+aware|notified)[^.!?]{0,160}?confirmed\s+that\s+(?:the\s+)?parts?\s+(?:was|were)\s+(?:not\s+yet\s+)?available[^.!?]*[.!?]? |
        \b(?:the\s+)?manufacturer\s+(?:was\s+)?(?:contacted|made\s+aware|notified)[^.!?]{0,160}?confirmed\s+that\s+(?:the\s+)?parts?\s+(?:was|were)\s+(?:not\s+yet\s+)?available[^.!?]*[.!?]? |
        \b(?:the\s+)?(?:local\s+|unknown\s+|undisclosed\s+)?dealer\s+(?:stated|confirmed|informed\s+the\s+contact)\s+that\s+(?:the\s+)?parts?\s+(?:was|were)\s+(?:not\s+yet\s+)?(?:available|unavailable|on\s+back\s*order|on\s+backorder)[^.!?]*[.!?]? |
        \b(?:the\s+)?part\s+was\s+(?:not\s+yet\s+)?available[.!?]? |
        \b(?:the\s+)?parts?\s+(?:was|were)\s+(?:not\s+yet\s+)?(?:available|unavailable|on\s+back\s*order|on\s+backorder)[.!?]? |
        \b(?:the\s+)?vin\s+was\s+(?:not\s+available|unavailable|invalid)[.!?]? |
        \binvalid\s+vin[.!?]? |
        \b(?:after\s+investigating\s+the\s+failure,\s+)?the\s+contact\s+(?:related|referenced|associated|linked|was\s+relating)\s+the\s+failures?\s+(?:to|with)\s+(?:nhtsa\s+campaign\s+number|customer\s+satisfaction\s+program|tsb)[:\s]*[^.!?]*[.!?]? |
        \bupon\s+investigation,\s+the\s+contact\s+(?:associated|linked|discovered)\s+[^.!?]{0,80}?(?:nhtsa\s+campaign\s+number|customer\s+satisfaction\s+program|tsb)[^.!?]*[.!?]? |
        \b(?:the\s+)?contact\s+researched\s+online\s+and\s+related\s+the\s+failures?\s+to\s+(?:nhtsa\s+campaign\s+number|customer\s+satisfaction\s+program|tsb)[^.!?]*[.!?]? |
        \b(?:the\s+)?vehicle\s+(?:was|had\s+been|had\s+not\s+been|had\s+yet\s+to\s+be|was\s+not\s+yet)\s+(?:not\s+)?(?:diagnosed|repaired|diagnosed\s+(?:or|nor)\s+repaired|taken\s+to\s+be\s+diagnosed\s+(?:or|nor)\s+repaired)[^.!?;]*[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+not\s+taken\s+to\s+(?:a\s+)?(?:local\s+)?dealer\s+or\s+(?:an\s+)?independent\s+mechanic[^.!?]*[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+not\s+taken\s+to\s+(?:a\s+)?dealer[^.!?]*[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+taken\s+to\s+(?:the\s+)?(?:local\s+)?dealer\s+to\s+be\s+diagnosed[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+taken\s+to\s+(?:an\s+)?independent\s+mechanic\s+to\s+be\s+diagnosed[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+towed\s+to\s+(?:the\s+)?(?:local\s+)?(?:dealer|residence|contact'?s\s+residence|tow\s+yard|tow\s+lot)[^.!?]*[.!?]? |
        \b(?:the\s+)?vehicle\s+remained\s+at\s+the\s+dealer\s+unrepaired[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+not\s+repaired\s+due\s+to\s+the\s+cost[.!?]? |
        \b(?:the\s+)?vehicle\s+was\s+not\s+repaired\s+and\s+remained\s+(?:at|in\s+the\s+possession\s+of)\s+the\s+dealer [.!?]? |
        \b(?:the\s+)?manufacturer\s+(?:was|had\s+been|had\s+not\s+been|had\s+yet\s+to\s+be|was\s+not\s+yet)\s+(?:not\s+)?(?:made\s+aware|notified|informed|contacted)(?:\s+of\s+(?:the\s+)?(?:failure|issue|recall))?[^.!?]*[.!?]? |
        \b(?:the\s+)?manufacturer\s+(?:was\s+also\s+)?notified\s+of\s+(?:the\s+)?(?:failure|issue)[^.!?]*[.!?]? |
        \b(?:the\s+)?manufacturer\s+was\s+contacted[^.!?]*[.!?]? |
        \b(?:the\s+)?(?:local\s+|unknown\s+|undisclosed\s+)?dealer\s+(?:was|had\s+been|was\s+not)\s+(?:not\s+)?(?:contacted|notified|made\s+aware|informed)(?:\s+of\s+(?:the\s+)?(?:failure|issue))?[^.!?]*[.!?]? |
        \b(?:the\s+)?(?:local\s+)?dealer\s+and\s+(?:the\s+)?manufacturer\s+were\s+(?:not\s+)?(?:contacted|notified|made\s+aware|informed)(?:\s+of\s+(?:the\s+)?(?:failure|issue))?[^.!?]*[.!?]? |
        \bneither\s+the\s+dealer\s+nor\s+the\s+manufacturer\s+were\s+notified\s+of\s+(?:the\s+)?(?:failure|issue)[^.!?]*[.!?]? |
        \b(?:no\s+additional\s+|no\s+further\s+)?assistance\s+was\s+(?:provided|offered)[.!?]? |
        \b(?:the\s+)?manufacturer\s+(?:provided|offered)\s+no\s+assistance[.!?]? |
        \b(?:the\s+)?manufacturer\s+was\s+notified\s+of\s+the\s+failure,\s+but\s+no\s+assistance\s+was\s+(?:provided|offered)[.!?]? |
        \b(?:the\s+)?manufacturer\s+was\s+notified\s+of\s+the\s+failure\s+but\s+(?:provided|offered)\s+no\s+assistance[.!?]? |
        \b(?:a\s+)?case\s+was\s+(?:opened|filed)[.!?]? |
        \b(?:the\s+)?manufacturer\s+(?:opened|filed)\s+a\s+case[.!?]? |
        \b(?:the\s+)?manufacturer\s+was\s+notified\s+of\s+the\s+failure\s+and\s+(?:a\s+)?case\s+was\s+(?:opened|filed)[.!?]? |
        \b(?:the\s+)?contact\s+was\s+(?:referred|advised)\s+to\s+(?:contact\s+)?the\s+nhtsa\s+hotline[^.!?]*[.!?]? |
        \b(?:the\s+)?manufacturer\s+referred\s+the\s+contact\s+to\s+the\s+nhtsa\s+hotline[^.!?]*[.!?]? |
        \b(?:the\s+)?manufacturer\s+was\s+made\s+aware\s+of\s+the\s+failure\s+and\s+referred\s+the\s+contact\s+to\s+(?:the\s+)?nhtsa\s+hotline[^.!?]*[.!?]? |
        \b(?:a|no)\s+police\s+report\s+was\s+(?:not\s+)?filed[.!?]? |
        \ba\s+police\s+report\s+was\s+taken\s+at\s+the\s+scene[.!?]? |
        \b(?:a\s+)?fire\s+(?:department\s+)?report\s+was\s+filed[.!?]? |
        \b(?:there\s+were\s+)?no\s+injur(?:y|ies)\s+(?:were\s+)?(?:sustained|reported)[.!?]? |
        \bno\s+medical\s+attention\s+was\s+required[.!?]? |
        \b(?:there\s+(?:was|were)\s+)?no\s+warning\s+(?:light|lights|lamp|lamps)\s+(?:was|were\s+)?illuminated[.!?]? |
        \bno\s+warning\s+lights?\s+illuminated[.!?]? |
        \bseveral\s+unknown\s+warning\s+(?:lights|lamps)\s+(?:were\s+)?illuminated[.!?]? |
        \ban\s+unknown\s+warning\s+light\s+was\s+illuminated[.!?]? |
        \b(?:the\s+)?cause\s+of\s+the\s+failure\s+was\s+not(?:\s+yet)?\s+determined[.!?]? |
        \bno\s+further\s+information\s+was\s+available[.!?]?
    )
    """,
    flags=re.I | re.X
)

ADMIN_LEADIN_ARTIFACTS_RE = re.compile(
    r"""
    \b(
        at\s+an?\s+(?:undisclosed|unknown)\s+speed |
        at\s+various\s+speeds |
        while\s+driving\s+at\s+an?\s+(?:undisclosed|unknown)\s+speed |
        (?:was\s+)?taken\s+to\s+(?:an?\s+)?(?:independent\s+)?(?:mechanic|dealer|local\s+dealer)(?:\s+who)?\s+(?:diagnosed|determined|informed\s+the\s+contact)\s+(?:that\s+)? |
        (?:an?\s+)?(?:independent\s+)?mechanic\s+(?:diagnosed|determined|informed\s+the\s+contact)\s+(?:that\s+)? |
        (?:the\s+)?dealer\s+(?:diagnosed|determined|informed\s+the\s+contact)\s+(?:that\s+)? |
        (?:the\s+)?contact\s+was\s+informed\s+(?:that\s+)? |
        (?:the\s+)?contact\s+was\s+advised\s+(?:that\s+)? |
        (?:the\s+)?contact\s+stated\s+(?:that\s+)? |
        (?:the\s+)?contact\s+also\s+stated\s+(?:that\s+)? |
        (?:the\s+)?contact\s+indicated\s+(?:that\s+)? |
        (?:the\s+)?contact\s+reported\s+(?:that\s+)? |
        the\s+failure\s+recurred |
        the\s+failure\s+reoccurred |
        the\s+failure\s+persisted |
        the\s+failure\s+was\s+intermittent |
        the\s+failure\s+had\s+been\s+recurring |
        the\s+failure\s+had\s+been\s+reoccurring |
        failure\s+became\s+a\s+regular\s+occurrence
    )\b[\s,:;-]*
    """,
    flags=re.I | re.X
)

DEALER_ADDRESS_RE = re.compile(
    r"""
    \(
        [^)]*(?:\b[A-Z]{2}\s*\d{5}(?:-\d{4})? |
            \bphone\s+number\b |
            \b\d{3}[-.)\s]*\d{3}[-.\s]*\d{4}\b |
            \b\d{3,5}\s+[A-Z0-9][A-Z0-9\s]{2,40}(?:rd|road|st|street|ave|avenue|blvd|boulevard|hwy|highway|dr|drive|ln|lane|pkwy|parkway)\b)
        [^)]*
    \)
    """,
    flags=re.I | re.X
)

RECALL_ARTIFACT_RE = re.compile(
    r"""
    (
        exceeded\s+a\s+reasonable\s+amount\s+of\s+time\s+for\s+the\s+recall\s+repair |
        vin\s+tool\s+confirms\s+parts?\s+not\s+available |
        parts?\s+distribution\s+disconnect |
        parts?\s+(?:was|were)\s+(?:not\s+yet\s+)?(?:available|unavailable|on\s+back\s*order|on\s+backorder) |
        vin\s+was\s+not\s+included\s+in\s+(?:the\s+recall|nhtsa\s+campaign) |
        nhtsa\s+campaign\s+number |
        recall\s+status\s+recall\s+incomplete\s+remedy\s+not\s+yet\s+available |
        \bhowever,\s+the\s+(?:part|parts)\s+to\s+do\s+the\s+recall\s+repairs?\s+(?:was|were)\s+(?:not\s+yet\s+)?(?:available|unavailable)[.!?]? |
        \b(?:the\s+)?contact\s+stated\s+that\s+the\s+manufacturer\s+(?:had\s+)?exceeded\s+a\s+reasonable\s+amount\s+of\s+time\s+for\s+the\s+recall\s+repairs?[.!?]? |
        \b(?:the\s+)?contact\s+was\s+informed\s+that\s+(?:the\s+)?(?:vehicle|vin)\s+was\s+not\s+included\s+in\s+(?:nhtsa\s+campaign\s+number[:\s]+[0-9A-Z]+[^.!?]*|the\s+recall)[.!?]? |
        \b(?:the\s+)?(?:vehicle|vin)\s+was\s+not\s+included\s+in\s+(?:nhtsa\s+campaign\s+number[:\s]+[0-9A-Z]+[^.!?]*|the\s+recall)[.!?]? |
        \b(?:the\s+)?contact\s+was\s+informed\s+that\s+(?:the\s+)?vin\s+was\s+not\s+under\s+recall[.!?]? |
        \bparts\s+not\s+(?:yet\s+)?available\b |
        \bconfirmed\s+that\s+parts\b.{0,30}\bnot\s+(?:yet\s+)?available\b |
        \brecall\b.{0,120}\b(?:part|parts)\b.{0,40}\bnot\s+(?:yet\s+)?available\b |
        \brecall\s+repair\b.{0,40}\bnot\s+(?:yet\s+)?available\b |
        \bpart\s+to\s+do\s+the\s+recall\s+repair\b.{0,40}\bnot\s+(?:yet\s+)?available\b |
        \bparts\s+distribution\s+disconnect\b |
        \bdistribution\s+disconnect\b |
        \bexceeded\s+a\s+reasonable\s+amount\s+of\s+time\b |
        \bexceeded\s+reasonable\s+time\b |
        \brecall\b.{0,160}\breasonable\s+amount\s+of\s+time\b |
        (?:part|parts)\s+to\s+do\s+the\s+recall\s+repairs?\s+(?:was|were)\s+(?:not\s+yet\s+)?(?:available|unavailable)[^.!?]*[.!?]?
    )
    """,
    flags=re.I | re.X
)

IN_OPERATION_RE = re.compile(
    r"""
    \b(
        while\s+driving |
        when\s+driving |
        while\s+operating |
        during\s+operation |
        while\s+the\s+vehicle\s+was\s+in\s+motion |
        while\s+in\s+motion |
        in\s+motion |
        while\s+traveling |
        while\s+driving\s+on |
        driving\s+on\s+the |
        driving\s+approximately |
        at\s+approximately\s+\d{1,3}\s*mph |
        approximately\s+\d{1,3}\s*mph |
        at\s+\d{1,3}\s*mph
    )\b
    """,
    flags=re.I | re.X
)

HIGHWAY_CONTEXT_RE = re.compile(
    r"""
    \b(
        highway |
        interstate |
        freeway |
        expressway |
        highway\s+speed |
        freeway\s+speed |
        interstate\s+speed
    )\b
    """,
    flags=re.I | re.X
)

SPEED_MPH_RE = re.compile(
    r'\b(?:approximately|approx\.?|about|around|at|going|traveling|driving)?\s*(\d{1,3})\s*mph\b', flags=re.I | re.X
)

CRITICAL_EVENT_RE = re.compile(
    r"""
    \b(
        stall(?:ed|ing)? |
        shut\s+off |
        turned\s+off |
        lost\s+(?:all\s+)?power |
        loss\s+of\s+(?:motive\s+)?power |
        lost\s+motive\s+power |
        would\s+not\s+accelerate |
        could\s+not\s+accelerate |
        failed\s+to\s+accelerate |
        brake(?:s)?\s+(?:failed|failure) |
        brake\s+pedal\s+(?:went\s+to\s+the\s+floor|soft|hard) |
        could\s+not\s+stop |
        unable\s+to\s+stop |
        could\s+not\s+steer |
        unable\s+to\s+steer |
        steering\s+(?:locked|seized) |
        caught\s+fire |
        fire |
        smoke |
        air\s*bags?\s+did\s+not\s+deploy |
        air\s*bags?\s+failed\s+to\s+deploy
    )\b
    """,
    flags=re.I | re.X
)

# Map contractions so we don't lose important negation when dropping punctuation
CONTRACTION_MAP = {
    "can't": "cannot",
    "can not": "cannot",
    "won't": "will not",
    "shan't": "shall not",

    "n't": " not",
    "'re": " are",
    "'ve": " have",
    "'ll": " will",
    "'d": " would",
    "'m": " am",
    "'s": "",
}


# Find max reported speed in pattern
def extract_max_mph(text):
    speeds = []
    for m in SPEED_MPH_RE.finditer(str(text)):
        with contextlib.suppress(ValueError):
            speeds.append(int(m.group(1)))
    if speeds:
        return max(speeds)
    return np.nan


# See if critical event happened while driving
def proximity_flag(text, pattern_a, pattern_b, window=120):
    text = str(text)
    a_spans = [m.span() for m in pattern_a.finditer(text)]
    b_spans = [m.span() for m in pattern_b.finditer(text)]

    for a_start, a_end in a_spans:
        for b_start, b_end in b_spans:
            if abs(a_start - b_start) <= window or abs(a_end - b_end) <= window:
                return True

    return False


# Finds where the FOIA text starts and removes everything after, almost always at the end of the text
def cut_foia_boilerplate(text):
    m = FOIA_START_RE.search(text)
    if m:
        return text[:m.start()]
    return text


# Removes any reviewer placed indicators in the text
def remove_nhtsa_markers(text):
    text = UPDATED_RE.sub(' ', text)
    text = LEADING_MARKER_RE.sub(' ', text)
    return TRAILING_MARKER_RE.sub(' ', text)


# Removes in leftover placeholders and personal info
def remove_personal_artifacts(text):
    for pattern, replacement in PERSONAL_ARTIFACT_PATTERNS:
        text = pattern.sub(replacement, text)
    return text


# Cuts out repeated third party phrases
def remove_all_admin_artifacts(text):
    for _ in range(2):
        text = DEALER_ADDRESS_RE.sub(' ', text)
        text = ADMIN_SENTENCE_ARTIFACTS_RE.sub(' ', text)
        text = ADMIN_PHRASES_RE.sub(' ', text)
        text = RECALL_ARTIFACT_RE.sub(' ', text)
        text = ADMIN_LEADIN_ARTIFACTS_RE.sub(' ', text)
        text = IN_OPERATION_RE.sub(' ', text)
        text = HIGHWAY_CONTEXT_RE.sub(' ', text)
        text = SPEED_MPH_RE.sub(' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
    return text


# Normalize any html, unicode, and non-standard spacing
def normalize_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\x00', ' ')
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Standardize apostrophe type then map contractions to expanded forms
def expand_contractions(text):
    text = str(text)
    text = text.replace("’", "'").replace("‘", "'").replace("`", "'")

    for contraction, expanded in CONTRACTION_MAP.items():
        if contraction.startswith("'") or contraction == "n't":
            text = re.sub(re.escape(contraction), expanded, text, flags=re.IGNORECASE)
        else:
            text = re.sub(rf"\b{re.escape(contraction)}\b", expanded, text, flags=re.IGNORECASE)

    return text


# Removes car make/mode/year if its in the complaint text
def remove_vehicle_identifiers(row):
    text = row['nlp_text']

    values_to_remove = [
        str(row.get('maketxt', '')),
        str(row.get('modeltxt', '')),
        str(row.get('yeartxt', '')),
    ]

    for value in values_to_remove:
        value = value.lower().strip()
        if value and value != 'nan':
            text = re.sub(rf'\b{re.escape(value)}\b', ' ', text)

    return re.sub(r'\s+', ' ', text).strip()


# Full text preprocessing cleanup
def clean_for_topic_modeling(text):
    text = normalize_text(text)
    text = expand_contractions(text)
    text = cut_foia_boilerplate(text)
    text = remove_nhtsa_markers(text)
    text = remove_personal_artifacts(text)
    text = remove_all_admin_artifacts(text)

    text = text.lower()
    text = re.sub(r'(\d+)(mph|mpg|mi|miles|km|rpm)', r'\1 \2', text)
    text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    # text = re.sub(r'\b\d+\b', ' ', text)

    return re.sub(r'\s+', ' ', text).strip()

df['in_operation_flag'] = (df['nlp_base'].str.contains(IN_OPERATION_RE, regex=True))
df['highway_flag'] = (df['nlp_base'].str.contains(HIGHWAY_CONTEXT_RE, regex=True))
df['max_reported_mph'] = df['nlp_base'].map(extract_max_mph)
df['high_speed_flag'] = (df['highway_flag'] | df['max_reported_mph'].ge(45).fillna(False))
df['recall_artifact_flag'] = df['nlp_base'].str.contains(RECALL_ARTIFACT_RE)
df['critical_event_near_operation_flag'] = df['nlp_base'].map(lambda x: proximity_flag(x, CRITICAL_EVENT_RE, IN_OPERATION_RE, window=160))

df['nlp_text'] = df['nlp_base'].map(clean_for_topic_modeling)
df['nlp_text'] = df.apply(remove_vehicle_identifiers, axis=1)
df['nlp_word_count'] = df['nlp_text'].str.split().str.len()
df = df[df['nlp_word_count'] >= 5].copy()

nlp_cols = [
    'odino',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'ldate',
    'state',
    'month',
    'severity_primary_flag',
    'severity_broad_flag',
    'component_groups',
    'raw_word_count',
    'nlp_raw',
    'nlp_word_count',
    'nlp_text',
    'in_operation_flag',
    'highway_flag',
    'high_speed_flag',
    'recall_artifact_flag',
    'critical_event_near_operation_flag'
]

nlp_df = df[nlp_cols].copy()
nlp_df.shape

C:\Users\davis\AppData\Local\Temp\ipykernel_928\3764311340.py:423: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['in_operation_flag'] = (df['nlp_base'].str.contains(IN_OPERATION_RE, regex=True))
C:\Users\davis\AppData\Local\Temp\ipykernel_928\3764311340.py:424: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['highway_flag'] = (df['nlp_base'].str.contains(HIGHWAY_CONTEXT_RE, regex=True))
C:\Users\davis\AppData\Local\Temp\ipykernel_928\3764311340.py:427: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['recall_artifact_flag'] = df['nlp_base'].str.contains(RECALL_ARTIFACT_RE)


(342049, 19)

## 4. Lemmatize The Complaint Text

Lemmatization reduces related word forms to a common base, which can help the topic model group similar complaints together. We keep the cleaned non-lemma text as well so later tables and clue terms stay easier to read.


In [ ]:
# Lemmatize the cleaned complaint text so the topic model groups related word forms together
import spacy

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

def lemmatize_text(text):
    doc = nlp(text)

    lemmas = []
    for token in doc:
        if token.is_space or token.is_punct:
            continue

        lemma = token.lemma_.lower().strip()

        if lemma == '-pron-':
            lemma = token.text.lower()

        lemmas.append(lemma)

    return ' '.join(lemmas)


def lemmatize_series(text_series, batch_size=1000, n_process=1):
    output = []

    for doc in nlp.pipe(
        text_series.fillna('').astype(str),
        batch_size=batch_size,
        n_process=n_process
    ):
        lemmas = []

        for token in doc:
            if token.is_space or token.is_punct:
                continue

            lemma = token.lemma_.lower().strip()

            if lemma == '-pron-':
                lemma = token.text.lower()

            lemmas.append(lemma)

        output.append(' '.join(lemmas))

    return output

nlp_df['nlp_text_lemma'] = lemmatize_series(nlp_df['nlp_text'])

## 5. Save Or Reload The Prepared NLP Table

The text preparation and lemmatization steps are the slowest part of the notebook. Save the prepared complaint table once, then reload it so the downstream topic model sections can be rerun much faster.


In [ ]:
# Save the prepared NLP table so later notebook runs can skip the long text preparation pass
nlp_df.to_parquet(NLP_PATH, index=False)

In [ ]:
# Reload the saved NLP table as the single source for the downstream topic model and watchlist sections
nlp_df = pd.read_parquet(NLP_PATH)

,odino,mfr_name,maketxt,modeltxt,yeartxt,ldate,state,month,severity_primary_flag,severity_broad_flag,...,nlp_text_lemma,max_reported_mph,in_operation_flag,highway_flag,high_speed_flag,recall_artifact_flag,critical_event_near_operation_flag,topic_survivor_unigram_count,recall_artifact_sparse_flag,topic_model_exclude_flag
0,10816121,"GENERAL MOTORS, LLC",PONTIAC,VIBE,2007,2020-10-05,OH,2020-10-01,False,False,...,passenger airbag complete the recall repair,NaN,False,False,False,True,False,4,False,False
1,11289434,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2019,2020-01-02,DE,2020-01-01,False,False,...,steer wheel control srs sub wire safety comple...,NaN,False,False,False,True,False,9,False,False
2,11289436,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,2018,2020-01-02,MA,2020-01-01,True,True,...,unintended acceleration the the while attempt ...,NaN,False,False,False,False,False,28,False,False
3,11290552,"CHRYSLER (FCA US, LLC)",DODGE,DURANGO,2011,2020-01-02,FL,2020-01-01,True,True,...,fire under the hood of vehicle the all the lig...,NaN,True,False,False,False,True,18,False,False
4,11292384,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ACCORD,2018,2020-01-01,PA,2020-01-01,False,False,...,drive at the car suddenly slow down from to 40...,70.0,False,True,True,False,False,72,False,False


## 6. Make Time-Based NLP Splits

Fit the topic model only on complaints through the end of 2024, then score later complaints forward in time. This keeps the topic library grounded in a development window and lets the watchlist behave like a forward monitor.


In [ ]:
# Split complaints by time so topics are fit on the development window and then scored forward into later months
train_df = nlp_df[nlp_df['ldate'] < pd.Timestamp('2025-01-01')].copy()
forward_df = nlp_df[nlp_df['ldate'] >= pd.Timestamp('2025-01-01')].copy()

## 7. Build The TF-IDF Text Matrix

Convert the prepared complaint text into weighted n-gram features. The stopword choices keep defect language while removing boilerplate words that blur topic meaning.


In [ ]:
# Build the custom stopword policy and vectorize only the text that should participate in topic modeling

# Words that are potentially useful in this context
PRESERVE_STOP_WORDS = {
    'no',
    'not',
    'never',
    'none',
    'without',
    'while',
    'after',
    'before',
    'during',
    'against',
    'off',
    'over',
    'under',
    'down',
    'up'
}

# Remove generic NHTSA words that are too common to be useful clues
NLP_EXTRA_STOP_WORDS = {
    'vehicle',
    'vehicles',
    'car',
    'contact',
    'owner',
    'consumer',
    'complainant',
    'driver',
    'stated',
    'informed',
    'told',
    'dealer',
    'dealership',
    'manufacturer',
    'repair',
    'repaired',
    'replace',
    'replacement',
    'reported',
    'reported',
    'noticed',
    'referenced',
    'notification',
    'notified',
    'stated',
    'states',
    'campaign',
    'unknown',
    'na',
    'nhtsa',
    'issue',
    'problem',
    'warranty',
    'service',
    'fix',
    'said',
    'known',
    'thank',
    'patience'
}

CUSTOM_STOP_WORDS = sorted((set(ENGLISH_STOP_WORDS) - PRESERVE_STOP_WORDS) | NLP_EXTRA_STOP_WORDS)

survivor_count_vectorizer = TfidfVectorizer(
    lowercase=False,
    stop_words=list(CUSTOM_STOP_WORDS),
    ngram_range=(1, 1),
    min_df=1
)
survivor_count_analyzer = survivor_count_vectorizer.build_analyzer()

nlp_df['topic_survivor_unigram_count'] = (
    nlp_df['nlp_text_lemma']
    .fillna('')
    .map(survivor_count_analyzer)
    .map(len)
)
nlp_df['recall_artifact_sparse_flag'] = (
    nlp_df['recall_artifact_flag'].fillna(False)
    & nlp_df['topic_survivor_unigram_count'].le(3)
)
nlp_df['topic_model_exclude_flag'] = nlp_df['recall_artifact_sparse_flag']

topic_filter_summary = pd.DataFrame([
    {
        'rows': len(nlp_df),
        'recall_artifact_rows': int(nlp_df['recall_artifact_flag'].sum()),
        'sparse_recall_artifact_rows': int(nlp_df['recall_artifact_sparse_flag'].sum()),
        'topic_model_rows_kept': int((~nlp_df['topic_model_exclude_flag']).sum())
    }
])
topic_filter_summary

train_df = nlp_df.loc[
    nlp_df['ldate'] < pd.Timestamp('2025-01-01')
].copy()
forward_df = nlp_df.loc[
    nlp_df['ldate'] >= pd.Timestamp('2025-01-01')
].copy()

train_df = train_df.loc[~train_df['topic_model_exclude_flag']].copy()
forward_df = forward_df.loc[~forward_df['topic_model_exclude_flag']].copy()

vectorizer = TfidfVectorizer(
    lowercase=False,
    stop_words=list(CUSTOM_STOP_WORDS),
    ngram_range=(1, 2),
    min_df=20,
    max_df=0.75,
    max_features=50000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df['nlp_text_lemma'])
X_forward = vectorizer.transform(forward_df['nlp_text_lemma'])
topic_feature_names = np.array(vectorizer.get_feature_names_out())

## 8. Scan Candidate Topic Counts

Try a small set of topic counts and compare them with simple diagnostics. Want to find a topic library that is both statistically reasonable and easy to interpret.


### Topic Count Scan

The scan below looks at how dominant the top topic is for each complaint, how evenly topics share the corpus, and how much the top terms overlap across topics.


In [ ]:
# Scan a small range of topic counts and compare them using dominance, balance, and overlap diagnostics
TOPIC_K_CANDIDATES = [12, 16, 20, 24]
TOPIC_STRONG_THRESHOLD = 0.20


def normalize_topic_weights(topic_weights):
    row_sums = topic_weights.sum(axis=1, keepdims=True)
    return np.divide(topic_weights, row_sums, out=np.zeros_like(topic_weights), where=row_sums > 0)


def get_topic_top_terms(model, feature_names, top_n=10):
    topic_term_map = {}
    for topic_id, weights in enumerate(model.components_):
        top_idx = np.argsort(weights)[::-1][:top_n]
        topic_term_map[topic_id] = feature_names[top_idx].tolist()
    return topic_term_map


scan_rows = []
for topic_k in TOPIC_K_CANDIDATES:
    print(f"Fitting for {topic_k} topics")
    topic_model = NMF(
        n_components=topic_k,
        init='nndsvda',
        solver='cd',
        random_state=RANDOM_SEED,
        max_iter=500
    )
    fit_weights = normalize_topic_weights(topic_model.fit_transform(X_train))

    dominant_topic = fit_weights.argmax(axis=1)
    dominant_share = pd.Series(dominant_topic).value_counts(normalize=True).reindex(range(topic_k), fill_value=0.0)

    top_terms = get_topic_top_terms(topic_model, topic_feature_names, top_n=10)
    overlaps = []
    for left_topic in range(topic_k):
        left_terms = set(top_terms[left_topic])
        for right_topic in range(left_topic + 1, topic_k):
            right_terms = set(top_terms[right_topic])
            union = left_terms | right_terms
            overlap = len(left_terms & right_terms) / len(union) if union else 0.0
            overlaps.append(overlap)

    scan_rows.append({
        'topic_k': topic_k,
        'fit_rows': len(train_df),
        'share_strong_dominant': float((fit_weights.max(axis=1) >= TOPIC_STRONG_THRESHOLD).mean()),
        'mean_max_topic_weight': float(fit_weights.max(axis=1).mean()),
        'max_topic_share': float(dominant_share.max()),
        'topics_under_1pct': int((dominant_share < 0.01).sum()),
        'mean_top10_jaccard_overlap': float(np.mean(overlaps)) if overlaps else 0.0,
        'median_top10_jaccard_overlap': float(np.median(overlaps)) if overlaps else 0.0
    })

topic_scan_df = pd.DataFrame(scan_rows)
topic_scan_df['topic_scan_score'] = (
    topic_scan_df['share_strong_dominant']
    - topic_scan_df['mean_top10_jaccard_overlap']
    - topic_scan_df['max_topic_share']
    - 0.02 * topic_scan_df['topics_under_1pct']
)

eligible_topic_scan_df = topic_scan_df.loc[
    (topic_scan_df['share_strong_dominant'] >= 0.75) &
    (topic_scan_df['topics_under_1pct'] <= 2) &
    (topic_scan_df['max_topic_share'] <= 0.35) &
    (topic_scan_df['mean_top10_jaccard_overlap'] < 0.35)
].copy()

topic_scan_rank_df = eligible_topic_scan_df if len(eligible_topic_scan_df) else topic_scan_df
recommended_topic_k = int(
    topic_scan_rank_df
    .sort_values(['topic_scan_score', 'topic_k'], ascending=[False, False])
    .iloc[0]['topic_k']
)

print(f"Development rows used for topic fitting: {len(train_df):,}")
print(f"Forward rows used for topic scoring: {len(forward_df):,}")
print(f"Recommended topic count from the scan: {recommended_topic_k}")
topic_scan_df

Fitting for 12 topics
Fitting for 16 topics
Fitting for 20 topics
Fitting for 24 topics
Development rows used for topic fitting: 258,329
Forward rows used for topic scoring: 75,078
Recommended topic count from the scan: 20


,topic_k,fit_rows,share_strong_dominant,mean_max_topic_weight,max_topic_share,topics_under_1pct,mean_top10_jaccard_overlap,median_top10_jaccard_overlap,topic_scan_score
0,12,258329,0.997890,0.549701,0.185411,0,0.008772,0.0,0.803708
1,16,258329,0.990864,0.493963,0.137232,0,0.006676,0.0,0.846956
2,20,258329,0.984469,0.477436,0.081923,0,0.008341,0.0,0.894206
3,24,258329,0.970642,0.439093,0.103558,0,0.007063,0.0,0.860021


### Lock The Final Topic Count

Once the scan is reviewed, lock the topic count so the downstream labels, categories, and watchlist groupings stay consistent.


In [ ]:
# Keep the scan result visible, but lock the final topic count so downstream labeling stays stable
TOPIC_K_LOCK = 20

print(f"Recommended topic count from the scan: {recommended_topic_k}")
print(f"Locked topic count for the final topic library: {TOPIC_K_LOCK}")

Recommended topic count from the scan: 20
Locked topic count for the final topic library: 20


## 9. Fit The Final Topic Model

Fit the final NMF model with the locked topic count, then assign each complaint to its strongest topic. This creates the complaint-topic table used by the rest of the notebook.


In [ ]:
# Fit the final NMF model, assign each complaint its strongest topic, and keep comparable outputs for development and forward periods
n_topics = TOPIC_K_LOCK

final_topic_model = NMF(
    n_components=n_topics,
    init='nndsvda',
    solver='cd',
    random_state=RANDOM_SEED,
    max_iter=500
)
topic_fit_weights = normalize_topic_weights(final_topic_model.fit_transform(X_train))
topic_forward_weights = normalize_topic_weights(final_topic_model.transform(X_forward))

topic_top_terms_map = get_topic_top_terms(final_topic_model, topic_feature_names, top_n=10)

fit_topic_id = topic_fit_weights.argmax(axis=1)
forward_topic_id = topic_forward_weights.argmax(axis=1)

fit_topic_weight_max = topic_fit_weights.max(axis=1)
forward_topic_weight_max = topic_forward_weights.max(axis=1)

train_df['topic_id'] = fit_topic_id
train_df['topic_strength'] = fit_topic_weight_max
train_df['topic_strength_flag'] = np.where(fit_topic_weight_max >= TOPIC_STRONG_THRESHOLD, 'strong', 'weak')

forward_df['topic_id'] = forward_topic_id
forward_df['topic_strength'] = forward_topic_weight_max
forward_df['topic_strength_flag'] = np.where(forward_topic_weight_max >= TOPIC_STRONG_THRESHOLD, 'strong', 'weak')
topic_df = pd.concat([train_df, forward_df], axis=0)

topic_library_rows = []
for topic_id in range(n_topics):
    topic_terms = topic_top_terms_map[topic_id]
    topic_examples = (
        train_df
        .assign(topic_weight=topic_fit_weights[:, topic_id])
        .sort_values('topic_weight', ascending=False)
        [['nlp_raw']]
        .drop_duplicates()
        .head(3)['nlp_raw']
        .fillna('')
        .astype(str)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .str.slice(0, 280)
        .tolist()
    )

    topic_library_rows.append({
        'topic_id': topic_id,
        'topic_top_terms': ' | '.join(topic_terms),
        'development_share': float((fit_topic_id == topic_id).mean()),
        'forward_share': float((forward_topic_id == topic_id).mean()) if len(forward_df) else 0.0,
        'median_topic_weight': float(np.median(topic_df.loc[topic_df['topic_id'].eq(topic_id), 'topic_strength'].dropna())),
        'example_complaint_1': topic_examples[0] if len(topic_examples) >= 1 else '',
        'example_complaint_2': topic_examples[1] if len(topic_examples) >= 2 else '',
        'example_complaint_3': topic_examples[2] if len(topic_examples) >= 3 else ''
    })

topic_library_base_df = pd.DataFrame(topic_library_rows)
topic_library_base_df['share_percent_change'] = (
    (topic_library_base_df['forward_share'] - topic_library_base_df['development_share'])
    / topic_library_base_df['development_share']
    * 100
)

print("Topic library preview:")
topic_library_base_df

Topic library preview:


,topic_id,topic_top_terms,development_share,forward_share,median_topic_weight,example_complaint_1,example_complaint_2,example_complaint_3,share_percent_change
0,0,time | drive | happen | not | no | start | up ...,0.076631,0.073284,0.344014,I have a wrangler 4Xe that I purchased August ...,This is the second time they have to replace t...,CAR BATTERY CONTINUES TO DIE AFTER ONLY 2 DAYS...,-4.367918
1,1,power | power steering | steering | assist | s...,0.044401,0.036082,0.436631,The car lost power steering while driving,Power steering went out. POWER STEERING ASSIST...,Electric power steering inop. POWER STEERING A...,-18.734529
2,2,engine | check | check engine | light | engine...,0.041768,0.051267,0.373582,The contact owns a 2014 Ford Escape. The conta...,Check engine light with code P052E.,Check engine light code P052E,22.740230
3,3,air bag | bag | air | bag light | bag not | pa...,0.031979,0.018701,0.484749,The contact owns a 2014 Buick Enclave. The con...,TL* THE CONTACT OWNS A 2008 BUICK ENCLAVE. THE...,The contact owns a 2014 GMC Acadia. The contac...,-41.521671
4,4,brake | pedal | brake pedal | stop | depress |...,0.071653,0.061429,0.430888,The actual brake pedal disintegrated.,Brake pedal go's to the floor. Spongy brakes,TL* THE CONTACT OWNS A 2008 FORD FUSION. THE C...,-14.267923
5,5,recall | not | safety | year | model | tell | ...,0.057125,0.085897,0.400759,I never got contacted about any recall on my v...,FOREST RIVER WILDWOOD. CONSUMER WRITES IN REGA...,I believe the manufacturer has exceeded a reas...,50.367773
6,6,transmission | shift | gear | park | shift gea...,0.058789,0.077426,0.445922,The contact owns a 2013 Ford F-150. The contac...,My car won’t shift into gear. It’s stuck in ne...,PROBLEM WITH TRANSMISSION. CHUGGING AND IRREGU...,31.700917
7,7,door | open | lock | door not | passenger | no...,0.037959,0.028823,0.545993,SLIDING DOOR LOCKS AND WON'T OPEN OR UNLOCK. D...,Doors are registered as open but are closed.,Left front drivers door flies open while drivi...,-24.067826
8,8,windshield | crack | windshield crack | crack ...,0.036186,0.025174,0.579128,MY 2017 SUBARU OUTBACK TOURING CRACKED MY WIND...,Windshield cracked across bottom portion. Many...,My front windshield has a crack and is now spr...,-30.432953
9,9,fuel | pump | tank | fuel pump | gasoline | fu...,0.041447,0.037987,0.471492,Fuel tank ventilation system malfunctioning,FUEL TANK DISTORTION FROM EXPANSION.,FUEL READS EMPTY WHEN THE TANK IS FULL. *TR,-8.347949


## 10. Label And Summarize The Topics

Turn raw topic ids into a readable topic library. This is where the notebook gives each topic a human label, a broader category, and a watchlist role for later reporting.


In [ ]:
# Map topic ids to human-readable labels, categories, and watchlist groups for all downstream reporting
topic_labels = {
    0: "Mixed driving incident / battery issues",
    1: "Electric power steering assist loss",
    2: "Check engine light / engine fault complaints",
    3: "Airbag warning / passenger airbag sensor issue",
    4: "Brake pedal / braking performance issue",
    5: "Recall notice / VIN eligibility / remedy process",
    6: "Transmission shifting / gear engagement issue",
    7: "Door latch / lock / unintended opening issue",
    8: "Windshield / glass cracking issue",
    9: "Fuel pump / fuel tank / leak or odor issue",
    10: "Steering wheel binding / difficult turning issue",
    11: "Airbag nondeployment / crash restraint issue",
    12: "Cruise / traction / electronic braking control issue",
    13: "Seat belt / passenger restraint issue",
    14: "Diagnosis / mechanic / repair follow-up cluster",
    15: "Oil consumption / low oil issue",
    16: "Coolant leak / cylinder intrusion / overheating issue",
    17: "Vehicle shutoff / stall while driving issue",
    18: "Headlight / low beam visibility issue",
    19: "Tire / suspension / frame rust mixed issue"
}

topic_categories = {
    0: 'defect_mixed',
    1: 'defect',
    2: 'defect',
    3: 'defect',
    4: 'defect',
    5: 'recall_process',
    6: 'defect',
    7: 'defect',
    8: 'defect',
    9: 'defect',
    10: 'defect',
    11: 'defect',
    12: 'defect',
    13: 'defect',
    14: 'defect_admin_mixed',
    15: 'defect',
    16: 'defect',
    17: 'defect',
    18: 'defect',
    19: 'defect_mixed'
}

defect_watchlist_topics = [
    1,
    2,
    3,
    4,
    6,
    7,
    8,
    9,
    10,
    13,
    15,
    16,
    17,
    18,
    19,
]
caution_topics = [
    0,
    14
]
process_topics = [5]

topic_df['topic_category'] = topic_df['topic_id'].map(topic_categories)
topic_df['topic_label'] = topic_df['topic_id'].map(topic_labels)
topic_df['watchlist_group'] = 'caution'
topic_df.loc[topic_df['topic_id'].isin(defect_watchlist_topics), 'watchlist_group'] = 'defect_watchlist'
topic_df.loc[topic_df['topic_id'].isin(process_topics), 'watchlist_group'] = 'process'

In [ ]:
# Pull together the complaint-level topic table with operational flags and representative text fields
flag_cols = [
    'severity_broad_flag',
    'severity_primary_flag',
    'in_operation_flag',
    'highway_flag',
    'high_speed_flag',
    'critical_event_near_operation_flag',
    'recall_artifact_flag'
]

flag_cols = [c for c in flag_cols if c in topic_df.columns]

for col in flag_cols:
    topic_df[col] = topic_df[col].fillna(False).astype(int)

topic_df['month'] = pd.to_datetime(topic_df['month']).dt.to_period('M').dt.to_timestamp()
topic_df['ldate'] = pd.to_datetime(topic_df['ldate'])

In [ ]:
# Aggregate complaint-level topic assignments into a library that is easy to review and compare
topic_library = (
    topic_df
    .groupby(
        ['topic_id', 'topic_label', 'topic_category', 'watchlist_group'],
        dropna=False
    )
    .agg(
        complaints=('odino', 'nunique'),
        avg_topic_strength=('topic_strength', 'mean'),
        median_topic_strength=('topic_strength', 'median'),
        broad_severity_rate=('severity_broad_flag', 'mean'),
        primary_severity_rate=('severity_primary_flag', 'mean'),
        in_operation_rate=('in_operation_flag', 'mean'),
        highway_rate=('highway_flag', 'mean'),
        high_speed_rate=('high_speed_flag', 'mean'),
        critical_in_operation_rate=('critical_event_near_operation_flag', 'mean'),
        recall_case_rate=('recall_artifact_flag', 'mean'),
        first_month=('month', 'min'),
        last_month=('month', 'max')
    )
    .reset_index()
    .sort_values("complaints", ascending=False)
)

rate_cols = [
    'broad_severity_rate',
    'primary_severity_rate',
    'in_operation_rate',
    'highway_rate',
    'high_speed_rate',
    'critical_in_operation_rate',
    'recall_case_rate'
]

topic_library[rate_cols] = (topic_library[rate_cols] * 100).round(2)
topic_library

,topic_id,topic_label,topic_category,watchlist_group,complaints,avg_topic_strength,median_topic_strength,broad_severity_rate,primary_severity_rate,in_operation_rate,highway_rate,high_speed_rate,critical_in_operation_rate,recall_case_rate,first_month,last_month
17,17,Vehicle shutoff / stall while driving issue,defect,defect_watchlist,26834,0.421788,0.395839,6.25,4.45,29.74,18.63,22.43,11.68,2.84,2020-01-01,2026-02-01
0,0,Mixed driving incident / battery issues,defect_mixed,caution,25298,0.373702,0.344014,7.29,7.04,23.53,28.11,33.35,3.12,0.49,2020-01-01,2026-02-01
4,4,Brake pedal / braking performance issue,defect,defect_watchlist,23122,0.463220,0.430888,14.66,11.24,30.97,10.49,16.44,4.17,11.59,2020-01-01,2026-02-01
19,19,Tire / suspension / frame rust mixed issue,defect_mixed,defect_watchlist,22412,0.485028,0.450724,7.73,5.91,23.70,13.40,19.25,0.91,4.71,2020-01-01,2026-02-01
14,14,Diagnosis / mechanic / repair follow-up cluster,defect_admin_mixed,caution,22383,0.448230,0.426449,30.71,5.56,62.16,6.57,25.57,29.43,27.66,2020-01-01,2026-02-01
5,5,Recall notice / VIN eligibility / remedy process,recall_process,process,21206,0.450374,0.400759,3.47,3.13,12.71,8.68,9.85,2.90,6.87,2020-01-01,2026-02-01
6,6,Transmission shifting / gear engagement issue,defect,defect_watchlist,21000,0.475313,0.445922,4.92,1.89,30.44,16.58,22.61,4.30,5.05,2020-01-01,2026-02-01
12,12,Cruise / traction / electronic braking control...,defect,caution,17894,0.460985,0.431142,4.49,3.75,30.08,19.20,26.13,2.60,5.12,2020-01-01,2026-02-01
15,15,Oil consumption / low oil issue,defect,defect_watchlist,16658,0.483789,0.449410,4.81,2.14,21.79,15.69,18.98,7.11,3.60,2020-01-01,2026-02-01
18,18,Headlight / low beam visibility issue,defect,defect_watchlist,16026,0.510511,0.455389,2.60,2.51,18.64,5.07,6.21,0.92,4.90,2020-01-01,2026-02-01


## 11. Track Global Topic Movement

Start with a high-level monthly topic view before splitting by components or cohorts. This makes it easier to see whether a theme is generally growing or if it only shows up inside a narrow vehicle slice.

In [ ]:
# Build helper functions that expand sparse month-level topic tables into complete monthly grids before rolling calculations
def complete_monthly_grid(df, group_cols, month_col='month'):
    months = pd.date_range(
        df[month_col].min(),
        df[month_col].max(),
        freq='MS'
    )

    groups = df[group_cols].drop_duplicates()
    month_df = pd.DataFrame({month_col: months})

    return groups.merge(month_df, how='cross')


def complete_group_months_since_first(df, group_cols, month_col='month'):
    max_month = df[month_col].max()
    month_df = pd.DataFrame({
        month_col: pd.date_range(df[month_col].min(), max_month, freq='MS')
    })
    group_start_df = (
        df.groupby(group_cols, dropna=False)[month_col]
        .min()
        .reset_index(name='first_observed_month')
    )

    group_month_df = group_start_df.merge(month_df, how='cross')
    group_month_df = group_month_df.loc[
        group_month_df[month_col] >= group_month_df['first_observed_month']
    ].copy()

    return group_month_df.drop(columns='first_observed_month').reset_index(drop=True)


def add_growth_metrics(df, group_cols, count_col='complaints', window=6):
    df = df.sort_values(group_cols + ['month']).copy()
    grouped = df.groupby(group_cols, dropna=False)[count_col]

    df[f'rolling_{window}mo_avg'] = grouped.transform(
        lambda s: s.shift(1).rolling(window, min_periods=3).mean()
    )
    df[f'rolling_{window}mo_sum'] = grouped.transform(
        lambda s: s.shift(1).rolling(window, min_periods=3).sum()
    )

    baseline = df[f'rolling_{window}mo_avg']
    df[f'growth_vs_{window}mo_avg'] = (df[count_col] / baseline.clip(lower=1))
    df[f'growth_delta_{window}mo'] = (df[count_col] - baseline.fillna(0))
    df['new_or_reemerging_signal'] = (
        (df[f'rolling_{window}mo_sum'].fillna(0) == 0)
        & (df[count_col] >= 3)
    )

    return df


def add_emerging_signal_flag(df, growth_threshold):
    df = df.copy()
    growth = df['growth_vs_6mo_avg'].replace([np.inf, -np.inf], np.nan)
    df['emerging_signal_flag'] = (
        growth.ge(growth_threshold).fillna(False)
        | df['new_or_reemerging_signal'].fillna(False)
    )
    return df


topic_group_cols = [
    'topic_id',
    'topic_label',
    'topic_category',
    'watchlist_group',
]

monthly_topic_raw = (
    topic_df
    .groupby(['month'] + topic_group_cols, dropna=False)
    .agg(
        complaints=('odino', 'nunique'),
        avg_topic_strength=('topic_strength', 'mean'),
        **{col.replace('_flag', '_rate'): (col, 'mean') for col in flag_cols}
    )
    .reset_index()
)

monthly_topic_grid = complete_monthly_grid(topic_df, topic_group_cols)

global_topic_monthly = monthly_topic_grid.merge(
    monthly_topic_raw,
    on=['month'] + topic_group_cols,
    how='left'
)

global_topic_monthly['complaints'] = global_topic_monthly['complaints'].fillna(0).astype(int)
global_topic_monthly['avg_topic_strength'] = global_topic_monthly['avg_topic_strength'].fillna(0)

for col in [c for c in global_topic_monthly.columns if c.endswith('_rate')]:
    global_topic_monthly[col] = global_topic_monthly[col].fillna(0)

global_topic_monthly = add_growth_metrics(
    global_topic_monthly,
    group_cols=topic_group_cols,
    window=6
)
global_topic_monthly = add_emerging_signal_flag(global_topic_monthly, growth_threshold=1.5)

## 12. Create A Ranking Score

The watchlist is not just about raw complaint counts. This section adds a ranking score so stronger rows rise to the top by combining volume, growth, severity/context clues, and confidence.


In [ ]:
# Score watchlist rows after eligibility is set so larger, faster growing, and higher risk signals rise to the top
def add_watchlist_score(df):
    df = df.copy()

    df['volume_score'] = np.log1p(df['complaints'])

    df['growth_score'] = (
        df['growth_vs_6mo_avg']
        .replace([np.inf, -np.inf], np.nan)
        .fillna(1)
        .clip(lower=0.5, upper=4.0)
    )

    df['new_signal_boost'] = np.where(df['new_or_reemerging_signal'], 1.35, 1.0)

    broad = df.get('severity_broad_rate', 0)
    primary = df.get('severity_primary_rate', 0)
    critical_op = df.get('critical_event_near_operation_rate', 0)
    in_op = df.get('in_operation_rate', 0)
    highway = df.get('highway_rate', 0)
    high_speed = df.get('high_speed_rate', 0)

    df['severity_multiplier'] = (1 + 0.75 * broad + 1.25 * primary)

    df['operational_risk_multiplier'] = (
        1
        + 0.20 * in_op
        + 0.30 * highway
        + 0.35 * high_speed
        + 0.60 * critical_op
    )

    df['confidence_multiplier'] = (0.75 + df['avg_topic_strength'].fillna(0).clip(0, 1))

    group_multiplier_map = {
        'defect_watchlist': 1.00,
        'caution': 0.65,
        'process': 0.45,
    }

    df['topic_group_multiplier'] = (
        df['watchlist_group']
        .map(group_multiplier_map)
        .fillna(0.60)
    )

    df['watchlist_score'] = (
        df['volume_score']
        * df['growth_score']
        * df['new_signal_boost']
        * df['severity_multiplier']
        * df['operational_risk_multiplier']
        * df['confidence_multiplier']
        * df['topic_group_multiplier']
    )

    return df


global_topic_monthly = add_watchlist_score(global_topic_monthly)
global_topic_monthly

,topic_id,topic_label,topic_category,watchlist_group,month,complaints,avg_topic_strength,severity_broad_rate,severity_primary_rate,in_operation_rate,...,new_or_reemerging_signal,emerging_signal_flag,volume_score,growth_score,new_signal_boost,severity_multiplier,operational_risk_multiplier,confidence_multiplier,topic_group_multiplier,watchlist_score
296,0,Mixed driving incident / battery issues,defect_mixed,caution,2020-01-01,521,0.397171,0.065259,0.065259,0.280230,...,True,True,6.257668,1.000000,1.35,1.130518,1.340979,1.147171,0.65,9.549646
297,0,Mixed driving incident / battery issues,defect_mixed,caution,2020-02-01,458,0.389538,0.072052,0.072052,0.292576,...,True,True,6.129050,1.000000,1.35,1.144105,1.301747,1.139538,0.65,9.127703
298,0,Mixed driving incident / battery issues,defect_mixed,caution,2020-03-01,310,0.384925,0.070968,0.070968,0.238710,...,True,True,5.739793,1.000000,1.35,1.141935,1.273548,1.134925,0.65,8.313183
299,0,Mixed driving incident / battery issues,defect_mixed,caution,2020-04-01,222,0.397408,0.067568,0.067568,0.211712,...,False,False,5.407172,0.516680,1.00,1.135135,1.250000,1.147408,0.65,2.956516
300,0,Mixed driving incident / battery issues,defect_mixed,caution,2020-05-01,270,0.404140,0.066667,0.055556,0.248148,...,False,False,5.602119,0.714758,1.00,1.119444,1.243704,1.154140,0.65,4.182181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661,19,Tire / suspension / frame rust mixed issue,defect_mixed,defect_watchlist,2025-10-01,324,0.443497,0.070988,0.064815,0.240741,...,False,False,5.783825,0.928810,1.00,1.134259,1.155556,1.193497,1.00,8.403625
662,19,Tire / suspension / frame rust mixed issue,defect_mixed,defect_watchlist,2025-11-01,296,0.436277,0.070946,0.060811,0.212838,...,False,False,5.693732,0.827201,1.00,1.129223,1.123480,1.186277,1.00,7.088247
663,19,Tire / suspension / frame rust mixed issue,defect_mixed,defect_watchlist,2025-12-01,334,0.423061,0.062874,0.047904,0.206587,...,False,False,5.814131,0.946176,1.00,1.107036,1.155539,1.173061,1.00,8.255120
664,19,Tire / suspension / frame rust mixed issue,defect_mixed,defect_watchlist,2026-01-01,366,0.433475,0.076503,0.060109,0.232240,...,False,False,5.905362,1.055769,1.00,1.132514,1.144945,1.183475,1.00,9.567594


## 13. Build Global Emerging-Topic Views

Create the broad global monitor and separate true emerging topic rows from process-heavy or purely contextual topics.


In [ ]:
# Build the global topic monitor first so we can see broad theme movement before drilling into components or cohorts
global_defect_topic_monitor = (
    global_topic_monthly
    .loc[
        (global_topic_monthly['watchlist_group'] == 'defect_watchlist')
        & (global_topic_monthly['complaints'] >= 10)
    ]
    .sort_values(['month', 'watchlist_score'], ascending=[False, False])
)

global_emerging_topics = (
    global_defect_topic_monitor
    .loc[global_defect_topic_monitor['emerging_signal_flag']]
    .copy()
)

global_process_monitor = (
    global_topic_monthly
    .loc[
        (global_topic_monthly['watchlist_group'] == 'process')
        & (global_topic_monthly['complaints'] >= 5)
    ]
    .sort_values(['month', 'watchlist_score'], ascending=[False, False])
)

global_emerging_topics

,topic_id,topic_label,topic_category,watchlist_group,month,complaints,avg_topic_strength,severity_broad_rate,severity_primary_rate,in_operation_rate,...,new_or_reemerging_signal,emerging_signal_flag,volume_score,growth_score,new_signal_boost,severity_multiplier,operational_risk_multiplier,confidence_multiplier,topic_group_multiplier,watchlist_score
802,6,Transmission shifting / gear engagement issue,defect,defect_watchlist,2025-03-01,561,0.502241,0.033868,0.014260,0.344029,...,False,True,6.331502,1.711235,1.00,1.043226,1.245900,1.252241,1.0,17.634622
504,15,Oil consumption / low oil issue,defect,defect_watchlist,2025-01-01,337,0.457586,0.026706,0.002967,0.201780,...,False,True,5.823046,1.653312,1.00,1.023739,1.195401,1.207586,1.0,14.227399
121,10,Steering wheel binding / difficult turning issue,defect,defect_watchlist,2023-12-01,422,0.505813,0.042654,0.028436,0.400474,...,False,True,6.047372,1.895210,1.00,1.067536,1.315284,1.255813,1.0,20.209289
931,1,Electric power steering assist loss,defect,defect_watchlist,2023-08-01,325,0.476085,0.046154,0.021538,0.387692,...,False,True,5.786897,1.737968,1.00,1.061538,1.301692,1.226085,1.0,17.039318
1301,16,Coolant leak / cylinder intrusion / overheatin...,defect,defect_watchlist,2023-08-01,301,0.548662,0.066445,0.009967,0.249169,...,False,True,5.710427,1.543590,1.00,1.062292,1.136213,1.298662,1.0,13.816566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184,13,Seat belt / passenger restraint issue,defect,defect_watchlist,2020-01-01,142,0.528217,0.119718,0.119718,0.211268,...,True,True,4.962845,1.000000,1.35,1.239437,1.126761,1.278217,1.0,11.959828
962,9,Fuel pump / fuel tank / leak or odor issue,defect,defect_watchlist,2020-01-01,270,0.519944,0.033333,0.022222,0.181481,...,True,True,5.602119,1.000000,1.35,1.052778,1.160926,1.269944,1.0,11.738482
1036,7,Door latch / lock / unintended opening issue,defect,defect_watchlist,2020-01-01,239,0.561871,0.054393,0.054393,0.158996,...,True,True,5.480639,1.000000,1.35,1.108787,1.078452,1.311871,1.0,11.606595
1332,8,Windshield / glass cracking issue,defect,defect_watchlist,2020-01-01,163,0.598097,0.006135,0.006135,0.319018,...,True,True,5.099866,1.000000,1.35,1.012270,1.197546,1.348097,1.0,11.251291


## 14. Explode Complaint Topics To Component Groups

Each complaint can mention more than one retained component group. Exploding those groups lets one topic be tracked across multiple component families without losing the original complaint-level topic assignment.


In [ ]:
# Split complaints into one row per retained component group so the same topic can be tracked across multiple component families
component_topic_df = topic_df.copy()

component_topic_df['component_group'] = (
    component_topic_df['component_groups']
    .fillna('')
    .str.split('|')
)

component_topic_df = component_topic_df.explode('component_group')

component_topic_df['component_group'] = (
    component_topic_df['component_group']
    .str.strip()
)

component_topic_df = component_topic_df.loc[
    component_topic_df['component_group'].ne('')
].copy()

component_topic_df

,odino,mfr_name,maketxt,modeltxt,yeartxt,ldate,state,month,severity_primary_flag,severity_broad_flag,...,topic_survivor_unigram_count,recall_artifact_sparse_flag,topic_model_exclude_flag,topic_id,topic_strength,topic_strength_flag,topic_category,topic_label,watchlist_group,component_group
0,10816121,"GENERAL MOTORS, LLC",PONTIAC,VIBE,2007,2020-10-05,OH,2020-10-01,0,0,...,4,False,False,11,0.727312,strong,defect,Airbag nondeployment / crash restraint issue,caution,AIR BAGS
1,11289434,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2019,2020-01-02,DE,2020-01-01,0,0,...,9,False,False,10,0.366289,strong,defect,Steering wheel binding / difficult turning issue,defect_watchlist,STEERING
2,11289436,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,2018,2020-01-02,MA,2020-01-01,1,1,...,28,False,False,4,0.446265,strong,defect,Brake pedal / braking performance issue,defect_watchlist,VEHICLE SPEED CONTROL
3,11290552,"CHRYSLER (FCA US, LLC)",DODGE,DURANGO,2011,2020-01-02,FL,2020-01-01,1,1,...,18,False,False,17,0.300829,strong,defect,Vehicle shutoff / stall while driving issue,defect_watchlist,ENGINE / COOLING
4,11292384,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ACCORD,2018,2020-01-01,PA,2020-01-01,0,0,...,72,False,False,0,0.370208,strong,defect_mixed,Mixed driving incident / battery issues,caution,ELECTRICAL SYSTEM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341994,11719264,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,HR-V,2020,2026-02-19,SD,2026-02-01,0,0,...,33,False,False,0,0.324958,strong,defect_mixed,Mixed driving incident / battery issues,caution,ELECTRICAL SYSTEM
341994,11719264,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,HR-V,2020,2026-02-19,SD,2026-02-01,0,0,...,33,False,False,0,0.324958,strong,defect_mixed,Mixed driving incident / battery issues,caution,WHEELS
341995,11719265,"NISSAN NORTH AMERICA, INC.",NISSAN,VERSA,2024,2026-02-19,MO,2026-02-01,0,0,...,68,False,False,0,0.279466,strong,defect_mixed,Mixed driving incident / battery issues,caution,POWER TRAIN
341996,11719266,"MERCEDES-BENZ USA, LLC",MERCEDES BENZ,SLK280,2007,2026-02-19,UT,2026-02-01,0,0,...,42,False,False,11,0.547388,strong,defect,Airbag nondeployment / crash restraint issue,caution,AIR BAGS


## 15. Build Component-Topic Monitors

Aggregate monthly signals at the component-topic level. This gives a middle layer between the broad global topic monitor and the final make/model/year cohort watchlist.


In [ ]:
# Aggregate topic signals at the component-topic-month level and create component watchlist and risk-monitor views
component_group_cols = [
    'component_group',
    'topic_id',
    'topic_label',
    'topic_category',
    'watchlist_group'
]

component_topic_raw = (
    component_topic_df
    .groupby(['month'] + component_group_cols, dropna=False)
    .agg(
        complaints=('odino', 'nunique'),
        avg_topic_strength=('topic_strength', 'mean'),
        **{col.replace('_flag', '_rate'): (col, 'mean') for col in flag_cols}
    )
    .reset_index()
)

component_topic_grid = complete_group_months_since_first(component_topic_df, component_group_cols)

component_topic_monthly = component_topic_grid.merge(
    component_topic_raw,
    on=['month'] + component_group_cols,
    how='left'
)

component_topic_monthly['complaints'] = component_topic_monthly['complaints'].fillna(0).astype(int)
component_topic_monthly['avg_topic_strength'] = component_topic_monthly['avg_topic_strength'].fillna(0)

for col in [c for c in component_topic_monthly.columns if c.endswith('_rate')]:
    component_topic_monthly[col] = component_topic_monthly[col].fillna(0)

component_topic_monthly = add_growth_metrics(
    component_topic_monthly,
    group_cols=component_group_cols,
    window=6
)
component_topic_monthly = add_emerging_signal_flag(component_topic_monthly, growth_threshold=1.75)
component_topic_monthly = add_watchlist_score(component_topic_monthly)

component_defect_topic_monitor = (
    component_topic_monthly
    .loc[
        (component_topic_monthly['watchlist_group'] == 'defect_watchlist')
        & (component_topic_monthly['complaints'] >= 5)
    ]
    .copy()
)

component_emerging_watchlist = (
    component_defect_topic_monitor
    .loc[component_defect_topic_monitor['emerging_signal_flag']]
    .sort_values(['month', 'watchlist_score'], ascending=[False, False])
)

component_risk_monitor = (
    component_defect_topic_monitor
    .loc[
        ~component_defect_topic_monitor['emerging_signal_flag']
        & (
            (component_defect_topic_monitor['severity_broad_rate'] >= 0.30)
            | (component_defect_topic_monitor['critical_event_near_operation_rate'] >= 0.20)
        )
    ]
    .sort_values(['month', 'watchlist_score'], ascending=[False, False])
)

component_emerging_watchlist

,component_group,topic_id,topic_label,topic_category,watchlist_group,month,complaints,avg_topic_strength,severity_broad_rate,severity_primary_rate,...,new_or_reemerging_signal,emerging_signal_flag,volume_score,growth_score,new_signal_boost,severity_multiplier,operational_risk_multiplier,confidence_multiplier,topic_group_multiplier,watchlist_score
4012,ELECTRICAL SYSTEM,8,Windshield / glass cracking issue,defect,defect_watchlist,2026-02-01,16,0.334462,0.125000,0.125000,...,False,True,2.833213,1.959184,1.00,1.25,1.078125,1.084462,1.0,8.112373
31679,VEHICLE SPEED CONTROL,18,Headlight / low beam visibility issue,defect,defect_watchlist,2026-02-01,7,0.284921,0.000000,0.000000,...,False,True,2.079442,2.470588,1.00,1.00,1.107143,1.034921,1.0,5.886510
12565,FUEL / PROPULSION,13,Seat belt / passenger restraint issue,defect,defect_watchlist,2026-02-01,5,0.302171,0.200000,0.200000,...,False,True,1.791759,1.764706,1.00,1.40,1.160000,1.052171,1.0,5.402868
32994,VISIBILITY / WIPER,16,Coolant leak / cylinder intrusion / overheatin...,defect,defect_watchlist,2026-02-01,5,0.291139,0.000000,0.000000,...,False,True,1.791759,1.875000,1.00,1.00,1.210000,1.041139,1.0,4.232287
23497,STEERING,3,Airbag warning / passenger airbag sensor issue,defect,defect_watchlist,2026-01-01,7,0.327445,0.714286,0.571429,...,False,True,2.079442,2.100000,1.00,2.25,1.307143,1.077445,1.0,13.837787
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22167,SERVICE BRAKES,6,Transmission shifting / gear engagement issue,defect,defect_watchlist,2020-01-01,5,0.373506,0.000000,0.000000,...,True,True,1.791759,1.000000,1.35,1.00,1.130000,1.123506,1.0,3.070912
12270,FUEL / PROPULSION,10,Steering wheel binding / difficult turning issue,defect,defect_watchlist,2020-01-01,5,0.286750,0.000000,0.000000,...,True,True,1.791759,1.000000,1.35,1.00,1.210000,1.036750,1.0,3.034402
24312,STEERING,15,Oil consumption / low oil issue,defect,defect_watchlist,2020-01-01,7,0.295558,0.000000,0.000000,...,True,True,2.079442,1.000000,1.35,1.00,1.028571,1.045558,1.0,3.018999
26014,STRUCTURE,18,Headlight / low beam visibility issue,defect,defect_watchlist,2020-01-01,5,0.274234,0.000000,0.000000,...,True,True,1.791759,1.000000,1.35,1.00,1.170000,1.024234,1.0,2.898667


## 16. Build Cohort-Topic Monitors

Move from broad component views to concrete vehicle cohorts. This is the main early-warning layer because it asks whether a specific make/model/year and topic combination is starting to stand out.


In [ ]:
# Build make/model/year cohort-topic signals so the final watchlist reflects concrete vehicle cohorts rather than only broad components
cohort_group_cols = [
    'maketxt',
    'modeltxt',
    'yeartxt',
    'component_group',
    'topic_id',
    'topic_label',
    'topic_category',
    'watchlist_group',
]

cohort_topic_raw = (
    component_topic_df
    .groupby(['month'] + cohort_group_cols, dropna=False)
    .agg(
        complaints=('odino', 'nunique'),
        avg_topic_strength=('topic_strength', 'mean'),
        **{col.replace('_flag', '_rate'): (col, 'mean') for col in flag_cols}
    )
    .reset_index()
)

In [ ]:
# Complete the cohort month grid from each group's first observed month, then merge the real monthly counts back in
cohort_topic_grid = complete_group_months_since_first(
    component_topic_df,
    cohort_group_cols
)

cohort_topic_monthly = cohort_topic_grid.merge(
    cohort_topic_raw,
    on=['month'] + cohort_group_cols,
    how='left'
)

cohort_topic_monthly['complaints'] = cohort_topic_monthly['complaints'].fillna(0).astype(int)
cohort_topic_monthly['avg_topic_strength'] = cohort_topic_monthly['avg_topic_strength'].fillna(0)

for col in [c for c in cohort_topic_monthly.columns if c.endswith('_rate')]:
    cohort_topic_monthly[col] = cohort_topic_monthly[col].fillna(0)

cohort_topic_monthly = add_growth_metrics(
    cohort_topic_monthly,
    group_cols=cohort_group_cols,
    window=6
)
cohort_topic_monthly = add_emerging_signal_flag(cohort_topic_monthly, growth_threshold=2.0)
cohort_topic_monthly = add_watchlist_score(cohort_topic_monthly)

cohort_defect_topic_monitor = (
    cohort_topic_monthly
    .loc[
        (cohort_topic_monthly['watchlist_group'] == 'defect_watchlist')
        & (cohort_topic_monthly['complaints'] >= 3)
    ]
    .copy()
)

cohort_emerging_watchlist = (
    cohort_defect_topic_monitor
    .loc[cohort_defect_topic_monitor['emerging_signal_flag']]
    .sort_values(['month', 'watchlist_score'], ascending=[False, False])
)

cohort_risk_monitor = (
    cohort_defect_topic_monitor
    .loc[
        ~cohort_defect_topic_monitor['emerging_signal_flag']
        & (
            (cohort_defect_topic_monitor['severity_broad_rate'] >= 0.35)
            | (cohort_defect_topic_monitor['critical_event_near_operation_rate'] >= 0.25)
            | (cohort_defect_topic_monitor['high_speed_rate'] >= 0.35)
        )
    ]
    .sort_values(['month', 'watchlist_score'], ascending=[False, False])
)

cohort_watchlist = cohort_emerging_watchlist.copy()

## 17. Build Final Watchlist Views

Turn the raw cohort-topic monitor into reader-facing views: detailed watchlist rows, a rolled-up summary, and the companion risk monitor.


In [ ]:
# Turn the final watchlist rows into short reviewer-facing reasons and simple signal tiers
def build_watchlist_reason(row):
    reasons = []

    if row.get('new_or_reemerging_signal', False):
        reasons.append(
            f"new/reemerging signal: {int(row['complaints'])} complaints after zero prior 6-mo baseline"
        )
    else:
        reasons.append(f"{row['growth_vs_6mo_avg']:.1f}x 6-mo baseline")

    if row.get('severity_broad_rate', 0) >= 0.35:
        reasons.append('high severity rate')

    if row.get('critical_event_near_operation_rate', 0) >= 0.25:
        reasons.append('critical event while in operation')

    if row.get('high_speed_rate', 0) >= 0.35:
        reasons.append('high-speed context')

    if row.get('avg_topic_strength', 0) >= 0.45:
        reasons.append('strong topic match')

    return '; '.join(reasons)


def build_monitor_reason(row):
    reasons = []

    if row.get('severity_broad_rate', 0) >= 0.35:
        reasons.append('high severity rate')

    if row.get('critical_event_near_operation_rate', 0) >= 0.25:
        reasons.append('critical event while in operation')

    if row.get('high_speed_rate', 0) >= 0.35:
        reasons.append('high-speed context')

    if row.get('avg_topic_strength', 0) >= 0.45:
        reasons.append('strong topic match')

    return '; '.join(reasons) if reasons else 'persistent risk context'


def assign_signal_tier(row):
    if row['complaints'] >= 10 and row['growth_vs_6mo_avg'] >= 2:
        return 'High-confidence signal'
    if row['complaints'] >= 5 and (
        row['growth_vs_6mo_avg'] >= 2
        or row['new_or_reemerging_signal']
    ):
        return 'Moderate signal'
    if row['complaints'] >= 3:
        return 'Early signal'
    return 'Monitor'


cohort_emerging_watchlist = cohort_emerging_watchlist.copy()
cohort_emerging_watchlist['watchlist_reason'] = cohort_emerging_watchlist.apply(build_watchlist_reason, axis=1)
cohort_emerging_watchlist['signal_tier'] = cohort_emerging_watchlist.apply(assign_signal_tier, axis=1)

cohort_watchlist_view = cohort_emerging_watchlist[
    [
        'month',
        'maketxt',
        'modeltxt',
        'yeartxt',
        'component_group',
        'topic_id',
        'topic_label',
        'complaints',
        'rolling_6mo_avg',
        'growth_vs_6mo_avg',
        'growth_delta_6mo',
        'severity_broad_rate',
        'critical_event_near_operation_rate',
        'in_operation_rate',
        'highway_rate',
        'high_speed_rate',
        'avg_topic_strength',
        'watchlist_score',
        'watchlist_reason',
        'signal_tier'
    ]
].copy()

cohort_watchlist_view['yeartxt'] = cohort_watchlist_view['yeartxt'].astype('str').fillna('Unknown')

cohort_risk_monitor = cohort_risk_monitor.copy()
cohort_risk_monitor['monitor_reason'] = cohort_risk_monitor.apply(build_monitor_reason, axis=1)

cohort_risk_monitor_view = cohort_risk_monitor[
    [
        'month',
        'maketxt',
        'modeltxt',
        'yeartxt',
        'component_group',
        'topic_id',
        'topic_label',
        'complaints',
        'rolling_6mo_avg',
        'growth_vs_6mo_avg',
        'growth_delta_6mo',
        'severity_broad_rate',
        'critical_event_near_operation_rate',
        'high_speed_rate',
        'avg_topic_strength',
        'watchlist_score',
        'monitor_reason'
    ]
].copy()

cohort_risk_monitor_view['yeartxt'] = cohort_risk_monitor_view['yeartxt'].astype('str').fillna('Unknown')

print("Emerging cohort watchlist:")
display(cohort_watchlist_view)

print("Persistent-risk cohort monitor:")
display(cohort_risk_monitor_view)

Emerging cohort watchlist:


,month,maketxt,modeltxt,yeartxt,component_group,topic_id,topic_label,complaints,rolling_6mo_avg,growth_vs_6mo_avg,growth_delta_6mo,severity_broad_rate,critical_event_near_operation_rate,in_operation_rate,highway_rate,high_speed_rate,avg_topic_strength,watchlist_score,watchlist_reason,signal_tier
2131101,2026-02-01,FORD,F-150,2017,POWER TRAIN,6,Transmission shifting / gear engagement issue,38,6.000000,6.333333,32.000000,0.026316,0.052632,0.500000,0.184211,0.263158,0.511001,24.877581,6.3x 6-mo baseline; strong topic match,High-confidence signal
2116149,2026-02-01,FORD,F-150,2015,POWER TRAIN,6,Transmission shifting / gear engagement issue,27,4.500000,6.000000,22.500000,0.000000,0.074074,0.259259,0.333333,0.444444,0.536379,23.178737,6.0x 6-mo baseline; high-speed context; strong...,High-confidence signal
2124013,2026-02-01,FORD,F-150,2016,POWER TRAIN,6,Transmission shifting / gear engagement issue,30,3.666667,8.181818,26.333333,0.000000,0.000000,0.333333,0.333333,0.466667,0.484119,22.545887,8.2x 6-mo baseline; high-speed context; strong...,High-confidence signal
4716820,2026-02-01,MAZDA,CX-90,2024,STEERING,10,Steering wheel binding / difficult turning issue,18,3.166667,5.684211,14.833333,0.000000,0.000000,0.166667,0.166667,0.166667,0.573995,17.802790,5.7x 6-mo baseline; strong topic match,High-confidence signal
3408631,2026-02-01,HYUNDAI,IONIQ 5,2024,ELECTRICAL SYSTEM,1,Electric power steering assist loss,7,0.500000,7.000000,6.500000,0.000000,0.714286,0.714286,0.428571,0.428571,0.355895,17.017361,7.0x 6-mo baseline; critical event while in op...,Moderate signal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2825932,2020-01-01,GMC,YUKON XL,2015,EXTERIOR LIGHTING,18,Headlight / low beam visibility issue,3,NaN,NaN,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.411664,2.174051,new/reemerging signal: 3 complaints after zero...,Early signal
5570523,2020-01-01,RAM,1500,2014,ENGINE / COOLING,16,Coolant leak / cylinder intrusion / overheatin...,3,NaN,NaN,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.410685,2.172218,new/reemerging signal: 3 complaints after zero...,Early signal
3074168,2020-01-01,HONDA,CR-V,2018,ENGINE / COOLING,15,Oil consumption / low oil issue,3,NaN,NaN,3.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.332983,2.161919,new/reemerging signal: 3 complaints after zero...,Early signal
4038488,2020-01-01,JEEP,RENEGADE,2018,ELECTRICAL SYSTEM,17,Vehicle shutoff / stall while driving issue,3,NaN,NaN,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.326477,2.014623,new/reemerging signal: 3 complaints after zero...,Early signal


Persistent-risk cohort monitor:


,month,maketxt,modeltxt,yeartxt,component_group,topic_id,topic_label,complaints,rolling_6mo_avg,growth_vs_6mo_avg,growth_delta_6mo,severity_broad_rate,critical_event_near_operation_rate,high_speed_rate,avg_topic_strength,watchlist_score,monitor_reason
2138330,2026-02-01,FORD,F-150,2018,POWER TRAIN,6,Transmission shifting / gear engagement issue,19,11.166667,1.701493,7.833333,0.000000,0.052632,0.368421,0.570762,9.088482,high-speed context; strong topic match
2100283,2026-02-01,FORD,F-150,2013,POWER TRAIN,6,Transmission shifting / gear engagement issue,7,4.333333,1.615385,2.666667,0.142857,0.000000,0.428571,0.437182,5.960428,high-speed context
3642163,2026-02-01,HYUNDAI,TUCSON,2025,FORWARD COLLISION AVOIDANCE,4,Brake pedal / braking performance issue,4,2.166667,1.846154,1.833333,0.000000,0.000000,0.500000,0.377492,4.438861,high-speed context
1982796,2026-02-01,FORD,EXPEDITION,2020,POWER TRAIN,6,Transmission shifting / gear engagement issue,4,3.000000,1.333333,1.000000,0.000000,0.000000,0.750000,0.351362,3.692862,high-speed context
1942522,2026-02-01,FORD,ESCAPE,2022,FUEL / PROPULSION,9,Fuel pump / fuel tank / leak or odor issue,3,2.333333,1.285714,0.666667,0.000000,0.333333,0.000000,0.695776,3.435894,critical event while in operation; strong topi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3787030,2020-04-01,JEEP,CHEROKEE,2016,POWER TRAIN,17,Vehicle shutoff / stall while driving issue,3,4.000000,0.750000,-1.000000,0.000000,0.333333,0.333333,0.474298,1.973039,critical event while in operation; strong topi...
2209773,2020-04-01,FORD,F-250,2019,STEERING,10,Steering wheel binding / difficult turning issue,5,11.000000,0.454545,-6.000000,0.000000,0.000000,1.000000,0.523283,1.836540,high-speed context; strong topic match
3791477,2020-04-01,JEEP,CHEROKEE,2017,ELECTRICAL SYSTEM,17,Vehicle shutoff / stall while driving issue,3,4.666667,0.642857,-1.666667,0.000000,0.333333,0.333333,0.533310,1.772691,critical event while in operation; strong topi...
5913912,2020-04-01,SUBARU,OUTBACK,2019,VISIBILITY / WIPER,8,Windshield / glass cracking issue,4,19.333333,0.206897,-15.333333,0.000000,0.000000,0.500000,0.787929,1.763581,high-speed context; strong topic match


In [ ]:
# Roll the raw cohort watchlist into a cleaner cohort-topic summary view without double-counting complaints
summary_group_cols = [
    'month',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'topic_id',
    'topic_label'
]

flagged_component_cols = summary_group_cols + [
    'component_group',
    'watchlist_score'
]

# Roll flagged component-topic rows back to unique complaints before summarizing
flagged_component_detail = component_topic_df.merge(
    cohort_emerging_watchlist[flagged_component_cols],
    on=flagged_component_cols[:-1],
    how='inner',
    suffixes=('', '_watchlist')
)

summary_rows = []
for group_key, group_df in flagged_component_detail.groupby(summary_group_cols, dropna=False):
    complaint_level_df = (
        group_df
        .sort_values('watchlist_score', ascending=False)
        .drop_duplicates('odino')
    )

    summary_rows.append({
        'month': group_key[0],
        'maketxt': group_key[1],
        'modeltxt': group_key[2],
        'yeartxt': group_key[3],
        'topic_id': group_key[4],
        'topic_label': group_key[5],
        'component_groups': ' | '.join(sorted(group_df['component_group'].dropna().astype(str).unique())),
        'complaints': int(complaint_level_df['odino'].nunique()),
        'max_component_watchlist_score': float(group_df['watchlist_score'].max()),
        'avg_topic_strength': float(complaint_level_df['topic_strength'].mean()),
        'severity_broad_rate': float(complaint_level_df['severity_broad_flag'].mean()),
        'critical_event_near_operation_rate': float(complaint_level_df['critical_event_near_operation_flag'].mean()),
        'highway_rate': float(complaint_level_df['highway_flag'].mean()),
        'high_speed_rate': float(complaint_level_df['high_speed_flag'].mean())
    })

cohort_topic_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(['month', 'max_component_watchlist_score'], ascending=[False, False])
    .reset_index(drop=True)
)

cohort_topic_summary

,month,maketxt,modeltxt,yeartxt,topic_id,topic_label,component_groups,complaints,max_component_watchlist_score,avg_topic_strength,severity_broad_rate,critical_event_near_operation_rate,highway_rate,high_speed_rate
0,2026-02-01,FORD,F-150,2017,6,Transmission shifting / gear engagement issue,ELECTRICAL SYSTEM | POWER TRAIN | VEHICLE SPEE...,39,24.877581,0.512759,0.025641,0.051282,0.179487,0.282051
1,2026-02-01,FORD,F-150,2015,6,Transmission shifting / gear engagement issue,POWER TRAIN,27,23.178737,0.536379,0.000000,0.074074,0.333333,0.444444
2,2026-02-01,FORD,F-150,2016,6,Transmission shifting / gear engagement issue,ELECTRICAL SYSTEM | ENGINE / COOLING | POWER T...,31,22.545887,0.499861,0.000000,0.000000,0.322581,0.451613
3,2026-02-01,MAZDA,CX-90,2024,10,Steering wheel binding / difficult turning issue,STEERING,18,17.802790,0.573995,0.000000,0.000000,0.166667,0.166667
4,2026-02-01,HYUNDAI,IONIQ 5,2024,1,Electric power steering assist loss,ELECTRICAL SYSTEM | POWER TRAIN,7,17.017361,0.355895,0.000000,0.714286,0.428571,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6051,2020-01-01,MAZDA,CX-5,2016,18,Headlight / low beam visibility issue,EXTERIOR LIGHTING,3,2.174721,0.412022,0.000000,0.000000,0.000000,0.000000
6052,2020-01-01,GMC,YUKON XL,2015,18,Headlight / low beam visibility issue,EXTERIOR LIGHTING,3,2.174051,0.411664,0.000000,0.000000,0.000000,0.000000
6053,2020-01-01,RAM,1500,2014,16,Coolant leak / cylinder intrusion / overheatin...,ENGINE / COOLING,3,2.172218,0.410685,0.000000,0.000000,0.000000,0.000000
6054,2020-01-01,JEEP,RENEGADE,2018,17,Vehicle shutoff / stall while driving issue,ELECTRICAL SYSTEM,3,2.014623,0.326477,0.000000,0.000000,0.000000,0.000000


## 18. Build The Recurring Large-Signal Companion Table

Not every useful signal is a one month spike. This final table keeps track of cohort-topic patterns that come back repeatedly or stay large over many months.


In [ ]:
# Summarize which cohort-topic signals keep returning across many months so persistent problems stay visible
recurring_group_cols = [
    'maketxt',
    'modeltxt',
    'yeartxt',
    'component_group',
    'topic_id',
    'topic_label'
]

signal_tier_order = {
    'Monitor': 0,
    'Early signal': 1,
    'Moderate signal': 2,
    'High-confidence signal': 3
}

recurring_large_signal_view = (
    cohort_watchlist_view
    .assign(signal_tier_rank=cohort_watchlist_view['signal_tier'].map(signal_tier_order).fillna(0))
    .groupby(recurring_group_cols, dropna=False)
    .agg(
        months_flagged=('month', 'nunique'),
        first_month=('month', 'min'),
        latest_month=('month', 'max'),
        max_complaints=('complaints', 'max'),
        avg_complaints=('complaints', 'mean'),
        median_complaints=('complaints', 'median'),
        max_growth_vs_6mo_avg=('growth_vs_6mo_avg', 'max'),
        avg_topic_strength=('avg_topic_strength', 'mean'),
        max_watchlist_score=('watchlist_score', 'max'),
        high_confidence_months=('signal_tier', lambda s: int((s == 'High-confidence signal').sum())),
        moderate_or_higher_months=('signal_tier', lambda s: int(s.isin(['Moderate signal', 'High-confidence signal']).sum())),
        best_signal_tier_rank=('signal_tier_rank', 'max')
    )
    .reset_index()
)

tier_rank_to_label = {value: key for key, value in signal_tier_order.items()}
recurring_large_signal_view['best_signal_tier'] = recurring_large_signal_view['best_signal_tier_rank'].map(tier_rank_to_label)
recurring_large_signal_view = recurring_large_signal_view.drop(columns='best_signal_tier_rank')

recurring_large_signal_view['avg_complaints'] = recurring_large_signal_view['avg_complaints'].round(2)
recurring_large_signal_view['median_complaints'] = recurring_large_signal_view['median_complaints'].round(2)
recurring_large_signal_view['max_growth_vs_6mo_avg'] = recurring_large_signal_view['max_growth_vs_6mo_avg'].round(2)
recurring_large_signal_view['avg_topic_strength'] = recurring_large_signal_view['avg_topic_strength'].round(3)
recurring_large_signal_view['max_watchlist_score'] = recurring_large_signal_view['max_watchlist_score'].round(3)

recurring_large_signal_view = recurring_large_signal_view.sort_values(
    ['months_flagged', 'max_complaints', 'max_watchlist_score'],
    ascending=[False, False, False]
).reset_index(drop=True)

recurring_large_signal_view

,maketxt,modeltxt,yeartxt,component_group,topic_id,topic_label,months_flagged,first_month,latest_month,max_complaints,avg_complaints,median_complaints,max_growth_vs_6mo_avg,avg_topic_strength,max_watchlist_score,high_confidence_months,moderate_or_higher_months,best_signal_tier
0,ACURA,RDX,2015,EXTERIOR LIGHTING,18,Headlight / low beam visibility issue,16,2020-12-01,2025-10-01,21,8.19,6.0,4.00,0.852,15.092,6,13,High-confidence signal
1,HONDA,CIVIC,2016,STEERING,10,Steering wheel binding / difficult turning issue,16,2020-02-01,2025-06-01,15,8.00,8.0,6.86,0.537,20.970,4,13,High-confidence signal
2,CHEVROLET,MALIBU,2009,STEERING,1,Electric power steering assist loss,16,2020-01-01,2025-02-01,12,5.50,4.5,5.00,0.475,13.239,0,8,Moderate signal
3,FORD,F-150,2019,POWER TRAIN,6,Transmission shifting / gear engagement issue,14,2020-02-01,2025-03-01,36,9.64,6.0,5.00,0.573,19.952,5,8,High-confidence signal
4,CADILLAC,SRX,2015,EXTERIOR LIGHTING,18,Headlight / low beam visibility issue,14,2020-01-01,2024-11-01,13,6.50,5.5,5.00,0.787,17.704,1,10,High-confidence signal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2877,TESLA,MODEL S,2015,ELECTRICAL SYSTEM,10,Steering wheel binding / difficult turning issue,1,2020-07-01,2020-07-01,3,3.00,3.0,NaN,0.280,1.927,0,0,Early signal
2878,FORD,E-350,2021,ENGINE / COOLING,2,Check engine light / engine fault complaints,1,2025-02-01,2025-02-01,3,3.00,3.0,NaN,0.270,1.910,0,0,Early signal
2879,CHEVROLET,SILVERADO 2500,2017,ELECTRICAL SYSTEM,16,Coolant leak / cylinder intrusion / overheatin...,1,2024-06-01,2024-06-01,3,3.00,3.0,NaN,0.234,1.842,0,0,Early signal
2880,SUBARU,ASCENT,2024,VISIBILITY / WIPER,19,Tire / suspension / frame rust mixed issue,1,2024-09-01,2024-09-01,3,3.00,3.0,NaN,0.220,1.816,0,0,Early signal


## 19. What To Review Next

The most useful outputs to inspect together are the topic library, the final watchlist summary, the risk monitor, and the recurring large-signal table. Read them as complaint-signal monitoring tools that help prioritize review, not as proof that a defect has been confirmed.
